# **Post-Earnings Announcement Drift Analysis**

### **Introduction**

Post-earnings drift is the market pattern in which a stock's price continues trending in the same direction as its earnings surprise for a period after the announcement. An earnings beat often extends into further gains beyond the initial jump, while an earnings miss can result in continued declines. This elaborates on the concept by highlighting the delayed market reaction and its role as a consistent, measurable input for strategy design.

The aim of this analysis is to find relevant signals for three stocks - Apple, Google and NVIDIA - that can help exploit post-earnings drift. To begin, we will be looking at two main signals: *Overreaction vs Underreaction*, and *Reaction Speed*. These are fundamental to post-earnings drift and will be built upon with other strong macroeconomic and financial metric factors that often impact drift. 

## **Importing the Data**

In [12]:
import pandas as pd
import numpy as np

# Load data
earnings_import = pd.read_csv("earnings_regression_panel.csv")
prices_import = pd.read_csv("event_study_price_data.csv")

# Inspect
earnings_import.head()

,Ticker,Earnings Date,Surprise,CAR,Regime,VIX,Δ10Y
0,GOOGL,2005-04-21,0.0095,0.000635,0,14.41,0.035971
1,NVDA,2005-05-12,0.0040,0.026393,0,16.12,0.027027
2,AAPL,2005-07-13,0.0000,0.001773,0,10.84,-0.047945
3,GOOGL,2005-07-21,0.0035,0.006394,0,10.97,-0.009259
4,NVDA,2005-08-11,0.0000,0.002138,0,12.42,0.033493


In [13]:
# now inspect the price data
prices_import.head()

,ticker,date,open,high,low,close,volume,earnings_date,td_num,event_day,daily_return,daily_return_pct
0,AAPL,2005-01-03,1.16,1.16,1.12,1.13,12382431,NaN,0,NaN,NaN,NaN
1,AAPL,2005-01-04,1.14,1.17,1.12,1.14,19608238,NaN,1,NaN,0.008850,0.884956
2,AAPL,2005-01-05,1.15,1.17,1.14,1.15,12157876,NaN,2,NaN,0.008772,0.877193
3,AAPL,2005-01-06,1.16,1.16,1.13,1.15,12604964,NaN,3,NaN,0.000000,0.000000
4,AAPL,2005-01-07,1.16,1.24,1.16,1.24,39923768,NaN,4,NaN,0.078261,7.826087


This data has been taken from someone else in the group. We will not be needing all the data, including CAR, which will compute ourselves and thus some columns will be deleted.

In [14]:
# --- Standardise column names (edit these mappings to match your actual file columns) ---
# Common patterns: 'ticker'/'Ticker', 'date'/'Date', 'close'/'Close' etc.

#Copy the original dataframes to avoid modifying them directly
earnings_v1 = earnings_import.copy()
prices_v1 = prices_import.copy()

# drop CAR and 10Y columns
earnings_v1 = earnings_import.drop(columns=["CAR", "Δ10Y"])

# Earnings table
earnings_v2 = earnings_v1.rename(columns={
    "Ticker": "ticker",
    "Earnings Date": "event_date",
    "Surprise": "surprise",
    "Regime": "regime"
})

earnings_v2["event_date"] = pd.to_datetime(earnings_v2["event_date"]).dt.normalize()
prices_v1["date"] = pd.to_datetime(prices_v1["date"]).dt.normalize()

prices_v1 = prices_v1.sort_values(["ticker", "date"])

#show the first few rows of the earnings data after renaming
earnings_v2.head()

,ticker,event_date,surprise,regime,VIX
0,GOOGL,2005-04-21,0.0095,0,14.41
1,NVDA,2005-05-12,0.0040,0,16.12
2,AAPL,2005-07-13,0.0000,0,10.84
3,GOOGL,2005-07-21,0.0035,0,10.97
4,NVDA,2005-08-11,0.0000,0,12.42


In [15]:
# drop the daily return columns if they exist in the price data (we will recalculate them later)
prices_v1 = prices_v1.drop(columns=["daily_return", "daily_return_pct"], errors="ignore")
prices_v1.head()

,ticker,date,open,high,low,close,volume,earnings_date,td_num,event_day
0,AAPL,2005-01-03,1.16,1.16,1.12,1.13,12382431,NaN,0,NaN
1,AAPL,2005-01-04,1.14,1.17,1.12,1.14,19608238,NaN,1,NaN
2,AAPL,2005-01-05,1.15,1.17,1.14,1.15,12157876,NaN,2,NaN
3,AAPL,2005-01-06,1.16,1.16,1.13,1.15,12604964,NaN,3,NaN
4,AAPL,2005-01-07,1.16,1.24,1.16,1.24,39923768,NaN,4,NaN


---

## **Cumulative Abnormal Returns calculation**

#### ***Methodology***

We calcualte *Abnormal Returns* and *Cumulative Abnormal Returns* as:

$$
AR_{i,t} = R_{i,t} - E(R_{i,t})
$$

$$
CAR_{i}(\tau_1, \tau_2) = \sum_{t=\tau_1}^{\tau_2} AR_{i,t}
$$

, respectively. 

Stock Return and Expected Stock Return are computed calculating the natural logarithm of the ratio of the stock's final price to its initial price. The logarithmic return is commonly used in financial analysis instead of the simple return (non-logarithmic). This is mainly because of the natural logarithm's *additive simplicity* - log returns can be summed to calculate total returns over multiple periods, while simple returns require multiplicative compounding. and *symmetry* - Log returns are more symmetric, meaning a 10% gain and a -10% loss are more balanced, which helps prevent bias in long-term performance modeling. This is well explained in this article: https://medium.com/@manojkotary/simple-returns-vs-log-returns-a-comprehensive-comparative-analysis-for-financial-analysis-702403693bad.

To summaries, it logarithmic returns produce much more accurate representations of return fluctuations and overall direction. The article is a great read.

### Computing Abnormal Returns

#### ***Methodology***

To compute the natural logarithmic market returns we will use the NASDAQ-100 ETF (QQQ) is used as the market proxy due to its strong exposure to large-cap technology stocks, aligning closely with the firms in our sample.

#### Importing NASDAQ 100 (QQQ) as market proxy and computing Market Returns

In [16]:
import pandas as pd
import numpy as np
import yfinance as yf

# Pull QQQ separately (daily data)
start_date = "2005-01-01"   # should cover your price dataset span
end_date = "2025-12-31"

qqq = yf.download("QQQ", start=start_date, end=end_date, auto_adjust=True, progress=False)
qqq = qqq.reset_index().rename(columns={"Date":"date", "Close":"close", "Adj Close":"adj_close", "Volume":"volume", "Open":"open", "High":"high", "Low":"low"})
qqq["date"] = pd.to_datetime(qqq["date"]).dt.normalize()

# Flatten MultiIndex columns if present
if isinstance(qqq.columns, pd.MultiIndex):
    qqq.columns = qqq.columns.get_level_values(0)

qqq.head()


Price,date,close,high,low,open,volume
0,2005-01-03,33.695377,34.369286,33.584480,34.198676,100970900
1,2005-01-04,33.081181,33.900109,32.884980,33.840392,136623200
2,2005-01-05,32.876450,33.234728,32.816737,32.995876,127925500
3,2005-01-06,32.714375,33.021472,32.705845,32.953230,102934600
4,2005-01-07,32.884979,33.157954,32.594943,32.893512,123104000


In [17]:
qqq.to_csv("qqq.csv", index=False)

In [18]:
# Compute daily log returns for QQQ
# we are adding a market return column to the QQQ dataframe
qqq["market_return"] = np.log(qqq["close"] / qqq["close"].shift(1))

# lets see the number of rows and columns in the QQQ dataframe
print(qqq.shape)
qqq.head()


(5282, 7)


Price,date,close,high,low,open,volume,market_return
0,2005-01-03,33.695377,34.369286,33.584480,34.198676,100970900,NaN
1,2005-01-04,33.081181,33.900109,32.884980,33.840392,136623200,-0.018396
2,2005-01-05,32.876450,33.234728,32.816737,32.995876,127925500,-0.006208
3,2005-01-06,32.714375,33.021472,32.705845,32.953230,102934600,-0.004942
4,2005-01-07,32.884979,33.157954,32.594943,32.893512,123104000,0.005201


### Computing Stock Returns

In [19]:
# copy the cleaned price data to a new dataframe for further processing
prices = prices_v1.copy()

# Compute daily log returns
# uses groupby and transform to calculate log returns for each ticker separately
prices["stock_return"] = prices.groupby("ticker")["close"].transform(
    lambda x: np.log(x / x.shift(1)) # this calculates the log return for each stock by taking the log of the current close price divided by the previous day's close price
)


prices.head()

,ticker,date,open,high,low,close,volume,earnings_date,td_num,event_day,stock_return
0,AAPL,2005-01-03,1.16,1.16,1.12,1.13,12382431,NaN,0,NaN,NaN
1,AAPL,2005-01-04,1.14,1.17,1.12,1.14,19608238,NaN,1,NaN,0.008811
2,AAPL,2005-01-05,1.15,1.17,1.14,1.15,12157876,NaN,2,NaN,0.008734
3,AAPL,2005-01-06,1.16,1.16,1.13,1.15,12604964,NaN,3,NaN,0.000000
4,AAPL,2005-01-07,1.16,1.24,1.16,1.24,39923768,NaN,4,NaN,0.075349


### Merge QQQ onto the Stock Data to compute CAR

In [20]:
# Make sure qqq has only date + market_return to merge cleanly
qqq_mkt = qqq[["date", "market_return"]].dropna().copy()

# Merge market returns into stock returns
prices = prices.merge(qqq_mkt, on="date", how="inner")

# Compute abnormal return (simple market-adjusted, no beta model)
prices["ar"] = prices["stock_return"] - prices["market_return"]

prices.head()


,ticker,date,open,high,low,close,volume,earnings_date,td_num,event_day,stock_return,market_return,ar
0,AAPL,2005-01-04,1.14,1.17,1.12,1.14,19608238,NaN,1,NaN,0.008811,-0.018396,0.027207
1,AAPL,2005-01-05,1.15,1.17,1.14,1.15,12157876,NaN,2,NaN,0.008734,-0.006208,0.014942
2,AAPL,2005-01-06,1.16,1.16,1.13,1.15,12604964,NaN,3,NaN,0.000000,-0.004942,0.004942
3,AAPL,2005-01-07,1.16,1.24,1.16,1.24,39923768,NaN,4,NaN,0.075349,0.005201,0.070148
4,AAPL,2005-01-10,1.25,1.26,1.21,1.23,31018490,NaN,5,NaN,-0.008097,-0.000519,-0.007578


### Computing CAR Reaction Windows

In [21]:
def align_to_next_trading_day(stock_dates: pd.Index, event_date: pd.Timestamp):
    """If event_date is not in stock_dates, push forward to the next available trading day."""
    if event_date in stock_dates:
        return event_date
    pos = stock_dates.searchsorted(event_date)
    if pos >= len(stock_dates):
        return None
    return stock_dates[pos]

def event_window_df(prices_ticker: pd.DataFrame, event_date: pd.Timestamp, L=10, R=10):
    """
    Returns a dataframe with columns: date, ar, rel_day for the event window.
    prices_ticker must be indexed by date and contain 'ar'.
    """
    idx = prices_ticker.index
    ed = align_to_next_trading_day(idx, event_date)
    if ed is None:
        return None

    t0 = idx.get_loc(ed)
    left = max(0, t0 - L)
    right = min(len(idx) - 1, t0 + R)

    win = prices_ticker.iloc[left:right+1].copy()
    win["rel_day"] = np.arange(left - t0, right - t0 + 1)
    win["event_date_adj"] = ed
    return win.reset_index()  # bring date back as column

def car_from_window(win: pd.DataFrame, a: int, b: int):
    return win.loc[win["rel_day"].between(a, b), "ar"].sum()


In [23]:
# Derive tickers from proces DataFrame to address later error
tickers = sorted(prices["ticker"].unique())

In [24]:
# Create a dict of ticker -> dataframe indexed by date for fast slicing
prices_by_ticker = {}
for t in tickers:
    df_t = prices[prices["ticker"] == t].copy()
    df_t = df_t.set_index("date").sort_index()
    prices_by_ticker[t] = df_t

rows = []


for _, r in earnings.iterrows():
    t = r["ticker"]
    ed = r["event_date"]

    win = event_window_df(prices_by_ticker[t], ed, L=20, R=20) # if you added more days to the window, increase R and L accordingly
    if win is None:
        continue

    CAR_01  = car_from_window(win, 0, 1)
    CAR_25  = car_from_window(win, 2, 5)
    CAR_05  = car_from_window(win, 0, 5)
    CAR_610 = car_from_window(win, 6, 10)
    CAR_1120 = car_from_window(win, 11, 20)
    CAR_020 = car_from_window(win, 0, 20)
    CAR_220 = car_from_window(win, 2, 20)
    CAR_520 = car_from_window(win, 5, 20)


    speed = np.nan
    if abs(CAR_05) > 1e-12:
        speed = abs(CAR_01) / abs(CAR_05)

    rows.append({
        "ticker": t,
        "event_date": ed,
        "event_date_adj": pd.to_datetime(win["event_date_adj"].iloc[0]).normalize(),
        "CAR_0_1": CAR_01,
        "CAR_2_5": CAR_25,
        "CAR_0_5": CAR_05,
        "CAR_6_10": CAR_610,
        "CAR_11_20": CAR_1120,
        "CAR_0_20": CAR_020,
        "CAR_2_20": CAR_220,
        "CAR_5_20": CAR_520,
        "Speed_01_over_05": speed,
        "VIX": r.get("VIX", np.nan) # this assumes your earnings dataframe has a 'vix' column; if not, it will just be NaN
    })

event_df = pd.DataFrame(rows).sort_values(["ticker", "event_date_adj"])
# save to csv for later use
event_df.to_csv("event_data.csv", index=False)
event_df.head()

NameError: name 'earnings' is not defined

In [ ]:
import matplotlib.pyplot as plt

def compute_caar(prices_by_ticker, earnings, ticker):
    all_windows = []

    for _, r in earnings[earnings["ticker"]==ticker].iterrows():
        df_t = prices_by_ticker[ticker]
        idx = df_t.index
        ed = r["event_date"]

        if ed not in idx:
            pos = idx.searchsorted(ed)
            if pos >= len(idx):
                continue
            ed = idx[pos]

        t0 = idx.get_loc(ed)
        L,R = 10,20
        left = max(0,t0-L)
        right = min(len(idx)-1,t0+R)

        win = df_t.iloc[left:right+1].copy()
        win["rel_day"] = np.arange(left - t0, right - t0 + 1)

        all_windows.append(win[["rel_day","ar"]])

    combined = pd.concat(all_windows)
    caar = combined.groupby("rel_day")["ar"].mean().cumsum()

    return caar

plt.figure(figsize=(10,6))
for t in tickers:
    caar = compute_caar(prices_by_ticker, earnings, t)
    plt.plot(caar.index, caar.values, label=t)

plt.axvline(0, linestyle='--')
plt.title("Average Cumulative Abnormal Return Around Earnings")
plt.xlabel("Relative Day")
plt.ylabel("CAAR")
plt.legend()
plt.show()


The CAAR plot illustrates a clear and economically meaningful reaction to earnings announcements across all three firms, though with notable differences in magnitude and persistence. NVIDIA exhibits the strongest and most sustained post-earnings abnormal performance, with both a pronounced immediate jump and continued upward drift in the subsequent days, suggesting potential underreaction or gradual information incorporation. Apple shows a more moderate but steady continuation pattern, while Google demonstrates a smaller initial reaction and weaker follow-through. The divergence in slopes across firms indicates that earnings response dynamics are not uniform within large-cap technology stocks, implying that any PEAD-based strategy should likely be ticker-specific rather than universally applied. While the average cumulative pattern suggests post-earnings continuation, further event-level analysis is required to determine whether this reflects systematic underreaction or is driven by a subset of large moves.

#### 2022-2025

In [ ]:
# Now lets filter our dataset to include data from 2022 onwards
event_df_22_25 = event_df[event_df["event_date_adj"] >= pd.to_datetime("2022-01-01")].copy()
#save this filtered dataset for later use
event_df_22_25.to_csv("event_data_2022_2025.csv", index=False)
event_df_22_25.head()

In [ ]:
# lets show CAAR(0,20) for the events in our filtered dataset (2022-2025)
plt.figure(figsize=(8,6))
for t in event_df_22_25["ticker"].unique():
    caar = compute_caar(prices_by_ticker, event_df_22_25, t)
    plt.plot(caar.index, caar.values, label=t)
plt.axvline(0, linestyle='--')
plt.title("Average Cumulative Abnormal Return (2022-2025)")
plt.xlabel("Relative Day")
plt.ylabel("CAAR")
plt.legend()
plt.show()

The CAAR profile over the 2022–2025 sub-period reveals a materially stronger and more differentiated earnings reaction compared to the longer sample. NVIDIA displays a pronounced pre-earnings run-up and a substantial post-announcement jump, followed by continued positive drift over the subsequent days, indicating persistent underreaction in a high-growth, high-volatility environment. Apple exhibits a modest but short-lived post-earnings uplift, with gains stabilising quickly, suggesting relatively efficient price incorporation. In contrast, Google shows a brief positive reaction around day 0 that gradually reverses in the days that follow, implying weaker continuation dynamics and a greater tendency toward mean reversion during this period. Overall, the sharper magnitude of movements — particularly for NVIDIA — suggests that earnings sensitivity intensified in the post-2022 regime, reinforcing the importance of conditioning PEAD strategies on both firm characteristics and broader market environments rather than assuming stable behaviour across time.

----

## **Overreaction vs Underreaction** 

#### ***Methodology***

“We define reaction speed as the proportion of the total five-day abnormal return that is incorporated within the first two trading days. A higher value indicates rapid information incorporation, while a lower value suggests gradual adjustment consistent with underreaction dynamics.”

In [ ]:
event_df["sign_01"] = np.sign(event_df["CAR_0_1"]) # this creates a new column 'sign_01' which is +1 if CAR(0,1) is positive, -1 if negative, and 0 if zero
event_df["sign_25"] = np.sign(event_df["CAR_2_5"]) # this creates a new column 'sign_25' which is +1 if CAR(2,5) is positive, -1 if negative, and 0 if zero

event_df["reversal"] = (event_df["sign_01"] * event_df["sign_25"] < 0).astype(int) # this creates a new column 'reversal' which is 1 if sign_01 and sign_25 have opposite signs (indicating a reversal) and 0 otherwise
event_df["continuation"] = (event_df["sign_01"] * event_df["sign_25"] > 0).astype(int) # this creates a new column 'continuation' which is 1 if sign_01 and sign_25 have the same sign (indicating a continuation) and 0 otherwise

event_df.groupby("ticker").agg(
    n=("CAR_0_1", "size"), # this counts the number of events for each ticker
    reversal_rate=("reversal", "mean"), # this calculates the average of the 'reversal' column for each ticker, which gives the proportion of events that are reversals
    continuation_rate=("continuation", "mean"), # this calculates the average of the 'continuation' column for each ticker, which gives the proportion of events that are continuations
    avg_CAR01=("CAR_0_1", "mean"), # this calculates the average of the 'CAR_0_1' column for each ticker
    avg_CAR25=("CAR_2_5", "mean") # this calculates the average of the 'CAR_2_5' column for each ticker
)

In [ ]:
event_df.head()

In [ ]:
import seaborn as sns

plt.figure(figsize=(8,6))
sns.scatterplot(data=event_df, x="CAR_0_1", y="CAR_2_5", hue="ticker")
plt.axhline(0, linestyle='--')
plt.axvline(0, linestyle='--')
plt.title("Initial Reaction vs Subsequent Drift")
plt.xlabel("CAR (0,1)")
plt.ylabel("CAR (2,5)")
plt.show()


summary = event_df.groupby("ticker")[["reversal","continuation"]].mean().reset_index()

summary_melt = summary.melt(id_vars="ticker", var_name="Type", value_name="Rate")

plt.figure(figsize=(8,6))
sns.barplot(data=summary_melt, x="ticker", y="Rate", hue="Type")
plt.title("Reversal vs Continuation Rates")
plt.show()



The scatter plot of CAR(0,1) against CAR(2,5) visually examines the relationship between the immediate earnings reaction and the subsequent drift window. The concentration of points in the upper-right and lower-left quadrants — where the signs of initial reaction and subsequent drift align — indicates a positive relationship between early movement and later continuation. This pattern is particularly pronounced for NVIDIA, which displays both larger dispersion and stronger directional clustering, suggesting more persistent momentum following earnings shocks. While there remains dispersion across all firms, the overall positive association is consistent with underreaction dynamics rather than systematic overreaction. Importantly, the variability in dispersion across firms highlights differences in reaction stability, implying that continuation strategies may yield stronger and more consistent results in certain tickers than others.


The reversal versus continuation rates provide direct event-level evidence on whether initial earnings reactions tend to persist or mean-revert. Across all three firms, continuation occurs more frequently than reversal, with Apple and NVIDIA exhibiting particularly strong continuation tendencies. This suggests that, on average, the market does not fully incorporate earnings information immediately, leading to short-term drift consistent with underreaction rather than overreaction. Google shows a narrower gap between continuation and reversal rates, indicating comparatively more balanced or efficient adjustment dynamics. Overall, the dominance of continuation across tickers supports the presence of a behavioural underreaction effect, reinforcing the viability of a post-earnings continuation strategy, albeit with varying strength across firms.

#### 2022-2025

In [ ]:
# now lets do the same analysis but for the filtered dataset (2022-2025)

event_df_22_25["sign_01"] = np.sign(event_df_22_25["CAR_0_1"])
event_df_22_25["sign_25"] = np.sign(event_df_22_25["CAR_2_5"])
event_df_22_25["reversal"] = (event_df_22_25["sign_01"] * event_df_22_25["sign_25"] < 0).astype(int)
event_df_22_25["continuation"] = (event_df_22_25["sign_01"] * event_df_22_25["sign_25"] > 0).astype(int) #this will be 1 if both signs are the same (both positive or both negative), indicating continuation
event_df_22_25.groupby("ticker").agg(
    n=("CAR_0_1", "size"),
    reversal_rate=("reversal", "mean"),
    continuation_rate=("continuation", "mean"),
    avg_CAR01=("CAR_0_1", "mean"),
    avg_CAR25=("CAR_2_5", "mean")
)

# now to plot the scatter plot of CAR(0,1) vs CAR(2,5) for the 2022-2025 dataset
plt.figure(figsize=(8,6))
sns.scatterplot(data=event_df_22_25, x="CAR_0_1", y="CAR_2_5", hue="ticker")
plt.axhline(0, linestyle='--')
plt.axvline(0, linestyle='--')
plt.title("Initial Reaction vs Subsequent Drift (2022-2025)")
plt.xlabel("CAR (0,1)")
plt.ylabel("CAR (2,5)")
plt.show()

summary_22_25 = event_df_22_25.groupby("ticker")[["reversal","continuation"]].mean().reset_index()
summary_22_25_melt = summary_22_25.melt(id_vars="ticker", var_name="Type", value_name="Rate")
plt.figure(figsize=(8,6))
sns.barplot(data=summary_22_25_melt, x="ticker", y="Rate", hue="Type")
plt.title("Reversal vs Continuation Rates (2022-2025)")
plt.show()



The 2022–2025 scatter plot reveals a more dispersed and heterogeneous relationship between the initial earnings reaction and subsequent drift compared to the full sample. While NVIDIA continues to show several observations in the continuation quadrants (upper-right and lower-left), the dispersion is materially wider, with more extreme positive and negative outcomes following initial reactions. Apple and Google display a weaker and less consistent directional clustering, with a noticeable number of points in the reversal quadrants. This suggests that in the more recent period — characterised by higher macro volatility and regime shifts — earnings reactions became less uniformly persistent and more sensitive to broader market dynamics. Continuation remains present, particularly for NVIDIA, but the signal appears noisier and less mechanically stable than in the full-period sample.

The reversal versus continuation rates for 2022–2025 show a clear divergence in behaviour across firms. NVIDIA exhibits a strong continuation bias, with continuation materially exceeding reversal, reinforcing its tendency toward post-earnings drift even in a higher-volatility regime. In contrast, Apple and Google display a reversal-dominant profile, with reversal rates exceeding continuation rates. This marks a notable shift from the full-sample results, where continuation was generally more prevalent across all firms. The evidence suggests that in the post-2022 environment, earnings reactions for Apple and Google were more prone to short-term overreaction or mean reversion, while NVIDIA maintained stronger underreaction dynamics.

Relative to the full dataset, the 2022–2025 period demonstrates greater behavioural divergence and regime sensitivity. In the full sample, continuation dominated across all three firms, supporting a broadly applicable PEAD continuation strategy. However, in the more recent sub-period, this effect weakens for Apple and Google, where reversal behaviour becomes more frequent. NVIDIA remains the exception, with continuation not only persisting but appearing amplified in magnitude. This shift implies that post-earnings dynamics are not structurally constant over time and are likely influenced by broader market volatility, macro tightening cycles, and changes in growth expectations. Strategically, this suggests that PEAD-style continuation trades may require ticker-specific calibration and regime conditioning rather than blanket application across large-cap technology stocks.

## **Reaction Speed Analysis**

#### ***Methodology***

- still needs re writing

The idea behind this is to look at the relationship reaction speed and the 'shock' (CAR[2,5]) after the initial reaction as well as the subsequent drift (CAR[5,20]) to see how reaction speeds may elongate/prolong drift. The main analysis is looking at the subsequent drift after the 5th trading day, as this is when the shock after the initial reaction (overreaction or underreaction) calm down and a proper reversion begins to 'normal' returns. 

In [ ]:
#Reaction Speed analysis (2022–2025)


# Basic sanity checks: make sure speed exists and is valid
event_df_22_25 = event_df_22_25.copy()
event_df_22_25 = event_df_22_25.dropna(subset=["Speed_01_over_05", "CAR_2_5"])

# now we drop outliers
# now lets remove the top two outlier events from the event_df and see how the distribution of reaction speeds changes, as well as the relationship between speed and subsequent drift.
event_df_no_outliers_22_25 = event_df_22_25[event_df_22_25["Speed_01_over_05"] < event_df_22_25["Speed_01_over_05"].nlargest(2).min()].copy()


# Visual: Distribution of reaction speed by ticker
plt.figure(figsize=(8,6))
sns.boxplot(data=event_df_no_outliers_22_25, x="ticker", y="Speed_01_over_05", hue="ticker")
plt.title("Distribution of Reaction Speed by Ticker (2022–2025)")
plt.xlabel("Ticker")
plt.ylabel("Reaction Speed = |CAR(0,1)| / |CAR(0,5)|")
plt.show()

# Create speed buckets (Slow / Medium / Fast) within each ticker
# Reason: speed distributions can differ across tickers, so bucketing within ticker avoids one ticker dominating.
event_df_no_outliers_22_25["speed_bucket"] = event_df_no_outliers_22_25.groupby("ticker")["Speed_01_over_05"].transform(
    lambda s: pd.qcut(s, q=3, labels=["Slow", "Medium", "Fast"])
)

# Visual: Average drift (CAR 2–5) by speed bucket (overall)
plt.figure(figsize=(8,6))
sns.barplot(data=event_df_no_outliers_22_25, x="speed_bucket", y="CAR_2_5", order=["Slow", "Medium", "Fast"])
plt.axhline(0, linestyle="--")
plt.title("Subsequent Drift by Reaction Speed Bucket (2022–2025)")
plt.xlabel("Reaction Speed Bucket")
plt.ylabel("Average CAR (2–5)")
plt.show()

# Visual: Drift by speed bucket and ticker (more actionable for strategy)
plt.figure(figsize=(10,6))
sns.barplot(
    data=event_df_no_outliers_22_25,
    x="ticker",
    y="CAR_2_5",
    hue="speed_bucket",
    hue_order=["Slow", "Medium", "Fast"]
)
plt.axhline(0, linestyle="--")
plt.title("Subsequent Drift by Ticker and Reaction Speed Bucket (2022–2025)")
plt.xlabel("Ticker")
plt.ylabel("Average CAR (2–5)")
plt.show()

# Visual: Continuation probability by speed bucket (overall)
# Continuation is already defined earlier; this plot shows whether slow reactions lead to more continuation.
plt.figure(figsize=(8,6))
sns.barplot(data=event_df_no_outliers_22_25, x="speed_bucket", y="continuation", order=["Slow", "Medium", "Fast"])
plt.title("Continuation Rate by Reaction Speed Bucket (2022–2025)")
plt.xlabel("Reaction Speed Bucket")
plt.ylabel("Continuation Rate")
plt.show()

# Visual: Speed vs subsequent drift scatter + trend line (overall)
plt.figure(figsize=(8,6))
sns.regplot(
    data=event_df_no_outliers_22_25,
    x="Speed_01_over_05",
    y="CAR_2_5",
    scatter=True,
        scatter_kws={"alpha": 0.7},
    line_kws={"color": "red"}
)
plt.axhline(0, linestyle="--")
plt.title("Reaction Speed vs Subsequent Drift (2022–2025)")
plt.xlabel("Reaction Speed = |CAR(0,1)| / |CAR(0,5)|")
plt.ylabel("CAR (2–5)")
plt.show()

# Same regplot but separated by ticker (helps show NVDA vs AAPL vs GOOGL differences)
g = sns.lmplot(
    data=event_df_no_outliers_22_25,
    x="Speed_01_over_05",
    y="CAR_2_5",
    col="ticker",
    height=4,
    aspect=1,
    scatter_kws={"alpha": 0.7},
    line_kws={"color": "red"},
    hue="ticker",
    hue_order=event_df_no_outliers_22_25["ticker"].unique()
)
g.fig.subplots_adjust(top=0.8)
g.fig.suptitle("Speed vs Drift by Ticker (2022–2025)")
plt.show()

# 8) Simple regression confirmation (does speed predict drift?)
# If the coefficient on Speed is negative, it supports the idea that slower reaction predicts stronger drift.
import statsmodels.api as sm

df_reg = event_df_no_outliers_22_25.dropna(subset=["Speed_01_over_05", "CAR_2_5"]).copy()
X = sm.add_constant(df_reg["Speed_01_over_05"])
y = df_reg["CAR_2_5"]
model = sm.OLS(y, X).fit(cov_type="HC3")
print(model.summary())


In [ ]:
# 0) Basic sanity checks
event_df_22_25 = event_df_22_25.copy()
event_df_22_25 = event_df_22_25.dropna(subset=["Speed_01_over_05", "CAR_5_20"])

# Remove top two speed outliers
event_df_no_outliers_22_25 = event_df_22_25[
    event_df_22_25["Speed_01_over_05"] <
    event_df_22_25["Speed_01_over_05"].nlargest(2).min()
].copy()


# 1) Distribution of reaction speed
plt.figure(figsize=(8,6))
sns.boxplot(data=event_df_no_outliers_22_25, x="ticker", y="Speed_01_over_05", hue="ticker")
plt.title("Distribution of Reaction Speed by Ticker (2022–2025)")
plt.xlabel("Ticker")
plt.ylabel("Reaction Speed = |CAR(0,1)| / |CAR(0,5)|")
plt.show()


# 2) Speed buckets
event_df_no_outliers_22_25["speed_bucket"] = (
    event_df_no_outliers_22_25.groupby("ticker")["Speed_01_over_05"]
    .transform(lambda s: pd.qcut(s, q=3, labels=["Slow", "Medium", "Fast"]))
)


# 3) Average drift (CAR 2–20) by speed bucket
plt.figure(figsize=(8,6))
sns.barplot(
    data=event_df_no_outliers_22_25,
    x="speed_bucket",
    y="CAR_5_20",
    order=["Slow", "Medium", "Fast"]
)
plt.axhline(0, linestyle="--")
plt.title("Subsequent Drift (5–20) by Reaction Speed Bucket (2022–2025)")
plt.xlabel("Reaction Speed Bucket")
plt.ylabel("Average CAR (5–20)")
plt.show()


# 4) Drift (5–20) by speed bucket and ticker
plt.figure(figsize=(10,6))
sns.barplot(
    data=event_df_no_outliers_22_25,
    x="ticker",
    y="CAR_5_20",
    hue="speed_bucket",
    hue_order=["Slow", "Medium", "Fast"]
)
plt.axhline(0, linestyle="--")
plt.title("Subsequent Drift (5–20) by Ticker and Speed Bucket (2022–2025)")
plt.xlabel("Ticker")
plt.ylabel("Average CAR (5–20)")
plt.show()





# 6) Speed vs drift (5–20) scatter + regression
plt.figure(figsize=(8,6))
sns.regplot(
    data=event_df_no_outliers_22_25,
    x="Speed_01_over_05",
    y="CAR_5_20",
    scatter=True,
    scatter_kws={"alpha": 0.7},
    line_kws={"color": "red"}
)
plt.axhline(0, linestyle="--")
plt.title("Reaction Speed vs Subsequent Drift (5–20) (2022–2025)")
plt.xlabel("Reaction Speed = |CAR(0,1)| / |CAR(0,5)|")
plt.ylabel("CAR (5–20)")
plt.show()


# 7) Separated by ticker
g = sns.lmplot(
    data=event_df_no_outliers_22_25,
    x="Speed_01_over_05",
    y="CAR_5_20",
    col="ticker",
    height=4,
    aspect=1,
    scatter_kws={"alpha": 0.7},
    line_kws={"color": "red"},
    hue="ticker",
    hue_order=event_df_no_outliers_22_25["ticker"].unique()
)
g.fig.subplots_adjust(top=0.8)
g.fig.suptitle("Speed vs Drift (5–20) by Ticker (2022–2025)")
plt.show()


# 8) Regression confirmation
df_reg = event_df_no_outliers_22_25.dropna(
    subset=["Speed_01_over_05", "CAR_5_20"]
).copy()

X = sm.add_constant(df_reg["Speed_01_over_05"])
y = df_reg["CAR_5_20"]

model = sm.OLS(y, X).fit(cov_type="HC3")
print(model.summary())


---

## **Reaction Speed** - DELETE

### Computing Reaction Speed

In [ ]:
event_df.groupby("ticker").agg(
    avg_speed=("Speed_01_over_05", "mean"),
    med_speed=("Speed_01_over_05", "median"),
    n=("Speed_01_over_05", "size")
)


In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(data=event_df, x="ticker", y="Speed_01_over_05")
plt.title("Distribution of Reaction Speed by Ticker")
plt.show()

plt.figure(figsize=(8,6))
sns.scatterplot(data=event_df, x="Speed_01_over_05", y="CAR_2_5", hue="ticker")
plt.axhline(0, linestyle='--')
plt.title("Reaction Speed vs Subsequent Drift")
plt.show()


In [ ]:
# it seems that one event is skewing our distribution.
# we need to remove it and see how the results change. We can identify it by looking at the tail of the event_df sorted by speed.

event_df_sorted = event_df.sort_values(by="Speed_01_over_05", ascending=False)
event_df_sorted.head()

In [ ]:
# lets look at this singular event in more detail, and show its CAR(0,20) profile
event_df_sorted.iloc[0][["ticker", "event_date_adj", "CAR_0_1", "CAR_2_5", "CAR_0_5", "CAR_6_10", "CAR_11_20", "CAR_0_20", "Speed_01_over_05"]]

# lets visualise the CAR profile of this event in more detail by plotting the cumulative abnormal return from day 0 to day 20 for this specific event.
# we can do this by extracting the event window for this specific event and plotting the cumulative sum of the abnormal returns over the window.

In [ ]:
# create visualisation of the CAR profile for the outlier event
outlier_event = event_df_sorted.iloc[0]
ticker = outlier_event["ticker"]
event_date_adj = outlier_event["event_date_adj"]
win_outlier = event_window_df(prices_by_ticker[ticker], event_date_adj, L=0, R=20)
win_outlier["CAR"] = win_outlier["ar"].cumsum()
plt.figure(figsize=(8,6))
sns.lineplot(data=win_outlier, x="rel_day", y="CAR")
plt.axhline(0, linestyle='--')
plt.title("CAR Profile for Outlier Event")
plt.show()

In [ ]:
# now lets remove the top two outlier events from the event_df and see how the distribution of reaction speeds changes, as well as the relationship between speed and subsequent drift.
event_df_no_outliers = event_df[event_df["Speed_01_over_05"] < event_df["Speed_01_over_05"].nlargest(2).min()].copy()


In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(data=event_df_no_outliers, x="ticker", y="Speed_01_over_05")
plt.title("Distribution of Reaction Speed by Ticker")
plt.show()

plt.figure(figsize=(8,6))
sns.scatterplot(data=event_df_no_outliers, x="Speed_01_over_05", y="CAR_2_5", hue="ticker")
plt.axhline(0, linestyle='--')
plt.title("Reaction Speed vs Subsequent Drift")
plt.show()

#### 2022-2025

---

## Pair with VIX

In [ ]:
# Create High/Low VIX buckets (median split across all events)
event_df = event_df.dropna(subset=["VIX"]).copy()
event_df["vix_bucket"] = np.where(event_df["VIX"] >= event_df["VIX"].median(), "HighVIX", "LowVIX")

event_df.groupby(["ticker", "vix_bucket"]).agg(
    n=("CAR_0_1","size"),
    reversal_rate=("reversal","mean"),
    continuation_rate=("continuation","mean"),
    avg_speed=("Speed_01_over_05","mean"),
    avg_CAR25=("CAR_2_5","mean")
).reset_index()


In [ ]:
event_df["vix_bucket"] = np.where(
    event_df["VIX"] >= event_df["VIX"].median(),
    "High VIX",
    "Low VIX"
)

plt.figure(figsize=(8,6))
sns.barplot(data=event_df, x="ticker", y="CAR_2_5", hue="vix_bucket")
plt.title("Post-Earnings Drift by VIX Regime")
plt.show()


---

## **Pair Trading Strategy**

### Are Apple, Google or NVIDIA strongly correlated to eachother ?

In [ ]:
import pandas as pd
import numpy as np
prices_pt = prices.copy()



# Overall correlation
corr = rets["AAPL_ret"].corr(rets["NVDA_ret"])
print("Correlation (daily log returns):", corr)




In [ ]:
# prices must include QQQ too: columns: date, AAPL, NVDA, QQQ

df = prices_pt[["date", "AAPL", "NVDA", "QQQ"]].copy()
df = df.sort_values("date")

# Log returns
df["AAPL_ret"] = np.log(df["AAPL"] / df["AAPL"].shift(1))
df["NVDA_ret"] = np.log(df["NVDA"] / df["NVDA"].shift(1))
df["MKT_ret"]  = np.log(df["QQQ"]  / df["QQQ"].shift(1))

# Market-adjusted AR (simple)
df["AAPL_AR"] = df["AAPL_ret"] - df["MKT_ret"]
df["NVDA_AR"] = df["NVDA_ret"] - df["MKT_ret"]

df = df.dropna(subset=["AAPL_AR", "NVDA_AR"])

corr_ar = df["AAPL_AR"].corr(df["NVDA_AR"])
print("Correlation (abnormal returns, market-adjusted):", corr_ar)


In [ ]:
window = 60  # ~3 months of trading days

rets["rolling_corr_60d"] = rets["AAPL_ret"].rolling(window).corr(rets["NVDA_ret"])

plt.figure(figsize=(9,5))
plt.plot(rets["date"], rets["rolling_corr_60d"])
plt.title("Rolling 60-Day Correlation: AAPL vs NVDA (Daily Log Returns)")
plt.xlabel("Date")
plt.ylabel("Rolling correlation")
plt.axhline(0, linewidth=1)
plt.show()
